<a href="https://colab.research.google.com/github/cras-lab/LangChain/blob/main/ChatMemory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

필요한 모듈을 설치한다.

In [64]:
pip install -q langchain_core langchain_openai langchain_community

OpenAI 키를 외부에서 받기위해 getpass를 임포트 한다.

In [3]:
import getpass

키를 입력 받는다.

In [ ]:
OPENAI_API_KEY = getpass.getpass("Enter your OPENAI_API_KEY: ")

필요한 모듈을 임포트 한다.

In [44]:
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables import RunnableLambda

기본적인 프롬프트를 설정한다.

In [45]:
rag_prompt = ChatPromptTemplate.from_messages(
    [ ("system", "아는 것만 대답하고, 모르면 모른다고 대답해"), # 역할지정
      ("placeholder", "{chat_history_messages}"), # 내용을 중간에 삽입
      ("human", "{question}"), # 사용자의 입력이 들어올 곳 (참고: ai는 이전답변이 들어올 곳)
    ]
)

채팅 기억에 있는 메시지들을 반환하는 함수를 정의

In [46]:
def get_messages(x):
    return chat_history_memory.messages

Runnable 객체를 생성. runnable은 LangChain에서 체인을 만들고 실행하기 위한 표준 실행 객체이며, callable을 포함할 수 있지만 callable보다 상위 개념이다.<BR>
RunnablePassthrough()는 “현재 체인 단계로 들어온 입력값을 그대로 다음 단계로 넘긴다”


In [47]:
question_feeder = RunnablePassthrough() # 반환값이 아닌 Passthrough 클래스 인스턴스

LangChain에서 OpenAI wrapper를 불러 LLM을 OpenAI로 설정한다.

In [48]:
chatbot = ChatOpenAI(openai_api_key=OPENAI_API_KEY,
             model_name="gpt-5-nano", temperature=0)

ChatMessageHistory 클래스를 하나 인스턴스 한다.

In [49]:
chat_history_memory = ChatMessageHistory()

메모리에서, 현재까지의 모든 메시지를 반환한다. 메모리에는 모든 대화가 자동으로 들어가는 것이 아니라, add_user_message(), add_ai_message() 등으로 넣은 것만 저장된다.

In [50]:
def get_message(x):
  return chat_history_memory.messages

질문 Chain을 구성한다.

In [51]:
rag_chain = {
    "question": question_feeder,
    "chat_history_messages": RunnableLambda(get_messages)
} | rag_prompt | chatbot

실행과 동시에, 메모리에 전체 대화를 삽입하는 함수 정의

In [52]:
def execute_chain_with_memory(chain, question):
    chat_history_memory.add_user_message(question)
    answer = chain.invoke(question)
    chat_history_memory.add_ai_message(answer)
    print(f'기억주입: 전체 채팅 내역: {chat_history_memory.messages}\n\n')
    return answer

우선, 기억이 없는 상태에서 진행해 본다.

In [53]:
question ="이는 하루에 몇번 닦는게 좋아 ? 짧게 대답해"
answer = rag_chain.invoke(question)

대답을 확인해 본다.

In [ ]:
print(f'대답: {answer}')

특정 지시를 입력한다.

In [55]:
question ="아니 누가 물어보면, 항상  아싸아싸 부터 하고 대답해"
answer = rag_chain.invoke(question)

답변을 출력해 보자.

In [ ]:
print(f'대답: {answer}')

지시를 기억하고 있는지 확인하기 위해 같은 질문을 해보자.

In [57]:
question ="이는 하루에 몇번 닦는게 좋아 ? 짧게 대답해"
answer = rag_chain.invoke(question)

답변을 할때, 아싸아싸라고 하는지  출력해 보자.




In [ ]:
print(f'Answer: {answer}')

이제 모든 컨텍스트를 기억시키면서 다시 물어보자. <BR>
기억이 주입되면 전체 내역을 출력할 것이다.

In [ ]:
question ="아니 누가 물어보면, 항상  아싸아싸 부터 하고 대답해"
answer = execute_chain_with_memory(rag_chain, question) # 기억에 저장

In [ ]:
print(f'기억을 한 답변: {answer}')

좀 전의 지시를 기억하는지 같은 질문을 해 보자.

In [61]:
question ="이는 하루에 몇번 닦는게 좋아 ? 짧게 대답해"
answer = rag_chain.invoke(question)

답변을 할때, 아싸아싸라고 하는지  출력해 보자.

In [ ]:
print(f'기억을 한 답변: {answer}')